
# Diagnostics workflow in ProbPipe

This notebook is a walkthrough of the current diagnostics design.
It shows the diagnostics API shipped for the 0.2.0 beta: fit a posterior,
attach diagnostics in place, then inspect results through `posterior.diagnostics`.

The public workflow should stay small:

```python
add_mcmc_diagnostics(posterior)
add_loo(posterior)

posterior.diagnostics.summary_table()
posterior.diagnostics.loo.elpd_loo
```

`add_loo(posterior)` is the user-facing LOO API when pointwise log likelihoods
already exist under `posterior._auxiliary/arviz/log_likelihood`. If they are
missing, pass `model=` and `data=` so `add_loo` can populate that ArviZ input
internally instead of requiring a separate public precomputation step.

Diagnostics use a two-tree layout inside `posterior._auxiliary`.
The split is based on who owns the data and how it is used:

```text
posterior._auxiliary
|-- arviz/          # ArviZ-compatible model/sample data and raw diagnostic inputs
`-- diagnostics/    # ProbPipe diagnostic outputs, warnings, and run metadata
```

`arviz/` contains data that follow ArviZ `DataTree` / `InferenceData` group
semantics: posterior draws, sample statistics, observed data, posterior
predictive draws, and pointwise log likelihoods. These are usually quantities
indexed by chain/draw and/or observation dimensions, and they are inputs that
ArviZ can consume directly for summaries, comparisons, or plots.

`diagnostics/` contains outputs produced by ProbPipe diagnostic runs and the
metadata needed to interpret those runs. Put a value here when it answers
"what did this diagnostic conclude?" rather than "what model/sample data did
the backend produce?" Examples include R-hat, ESS, MCSE, LOO scalar
summaries, pass/fail flags, warnings,
timestamps, backend names, plotting readiness flags, and cached pointwise
diagnostic outputs such as Pareto-k or `loo_i`.

The practical rule is: raw ArviZ-shaped inputs live under `arviz/`; ProbPipe's
computed diagnostic results live under `diagnostics/`. Observation-level shape
alone does not determine placement. For example, pointwise log likelihoods are
`arviz/log_likelihood` because they are raw ArviZ inputs, while pointwise
Pareto-k values are `diagnostics/runs/loo` because they are results of the LOO
diagnostic.

This workflow demonstrates the main diagnostics path:

1. `condition_on` creates a posterior with ArviZ-compatible auxiliary data.
2. `add_mcmc_diagnostics` writes R-hat, ESS, MCSE, and warnings.
3. `add_loo` should own the log-likelihood check/computation path and write PSIS-LOO summaries.
4. Users read results through `posterior.diagnostics`, not by manually walking `_auxiliary`.



## 1. Setup

The setup imports the current diagnostics public API and a small helper for
printing the `_auxiliary` tree. 


In [1]:
import warnings
warnings.filterwarnings("ignore", message=r"Explicitly requested dtype.*float64.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.glm as tfp_glm

import probpipe.diagnostics as diagnostics
importlib.reload(diagnostics)

from probpipe.diagnostics import (
    add_mcmc_diagnostics,
    add_rhat,
    add_ess,
    add_mcse,
    add_loo,
)

from probpipe import (
    Record,
    NumericRecordArray,
    EventTemplate,
    Normal,
    ProductDistribution,
    EmpiricalDistribution,
    BootstrapReplicateDistribution,
    GLMLikelihood,
    SimpleModel,
    condition_on,
    mean,
    variance,
    workflow_function,
    provenance_ancestors,
)

from probpipe.modeling import IncrementalConditioner
from probpipe.custom_types import ArrayLike

In [2]:
# Helper: inspect _auxiliary layout

def has_path(tree, path):
    """Check whether a slash-separated path exists in a DataTree-like object."""
    if tree is None:
        return False

    node = tree
    for part in path.strip("/").split("/"):
        if not part:
            continue
        try:
            node = node[part]
        except Exception:
            return False

    return True


def show_auxiliary_paths(posterior):
    """Print expected diagnostics and ArviZ paths."""
    aux = getattr(posterior, "_auxiliary", None)

    print("posterior._auxiliary is None:", aux is None)

    paths = [
        "arviz",
        "arviz/posterior",
        "arviz/sample_stats",
        "arviz/observed_data",
        "arviz/posterior_predictive",
        "arviz/log_likelihood",
        "diagnostics",
        "diagnostics/mcmc",
        "diagnostics/runs",
        "diagnostics/runs/ppc",
        "diagnostics/runs/loo",
        "diagnostics/runs/spc",
    ]

    for path in paths:
        print(f"{path:35s}", has_path(aux, path))


## 2. Data

We use the horseshoe crabs data. The response is the number of satellites, and
the predictor is crab width. 


In [3]:
# ── Data ──────────────────────────────────────────────────────────────────────
data_path = next(
    path for path in (
        Path("../tutorials/data/horseshoe_crabs.csv"),
        Path("docs/tutorials/data/horseshoe_crabs.csv"),
    )
    if path.exists()
)
df = pd.read_csv(data_path)

df.head()
print(df)




     width_cm  satellites
0        28.3           8
1        22.5           0
2        26.0           9
3        24.8           0
4        26.0           4
..        ...         ...
167      32.5           0
168      23.5           0
169      26.5           6
170      25.5           3
171      22.5           1

[172 rows x 2 columns]


In [4]:
# Standardize width and assemble a NumericRecordArray with fields X and y.

@workflow_function
def prep_data(width: ArrayLike, satellites: ArrayLike) -> NumericRecordArray:
    width = np.asarray(width, dtype=np.float32)
    width_z = ((width - np.mean(width)) / np.std(width)).astype(np.float32)

    y = np.asarray(satellites, dtype=np.float32)

    template = EventTemplate(X=(), y=())

    return NumericRecordArray(
        {"X": width_z, "y": y},
        batch_shape=(len(width),),
        template=template,
    )


data = prep_data(df["width_cm"], df["satellites"])

print(data)
print("source:", data.source)

NumericRecordArray(batch_shape=(172,), X=array(shape=(172,)), y=array(shape=(172,)))
source: Provenance('workflow.prep_data', parents=[])



## 3. Poisson regression model

We fit a Poisson regression model for count data:

$$
y_i \sim \operatorname{Poisson}\left(\exp(\eta_i)\right),
\qquad
\eta_i = \alpha + x_i \beta.
$$

`GLMLikelihood` wraps a TFP GLM family and a design matrix into a ProbPipe
likelihood. After fitting, inspect the `_auxiliary` paths to show that the
posterior has an ArviZ-compatible subtree before any diagnostic summaries are
added.


In [5]:
lik_poisson = GLMLikelihood(tfp_glm.Poisson(), data["X"])

prior = ProductDistribution(
    Normal(loc=0.0, scale=jnp.sqrt(5.0), name="intercept"),
    Normal(loc=0.0, scale=jnp.sqrt(5.0), name="slope"),
)

model_poisson = SimpleModel(prior, lik_poisson, name="poisson")

posterior = condition_on(model_poisson, data["y"])

print(posterior)

ApproximateDistribution(algorithm='blackjax_nuts', num_chains=1, num_draws=1000, event_shapes={'intercept': (), 'slope': ()})


In [6]:
# Inspect auxiliary structure immediately after inference.
# Ideally, condition_on has populated the ArviZ-compatible subtree.

show_auxiliary_paths(posterior)

posterior._auxiliary is None: False
arviz                               True
arviz/posterior                     True
arviz/sample_stats                  True
arviz/observed_data                 False
arviz/posterior_predictive          False
arviz/log_likelihood                False
diagnostics                         False
diagnostics/mcmc                    False
diagnostics/runs                    False
diagnostics/runs/ppc                False
diagnostics/runs/loo                False
diagnostics/runs/spc                False



## 4. MCMC diagnostics

This section demonstrates the first diagnostics layer: convergence and Monte
Carlo accuracy checks for posterior draws.

```python
add_mcmc_diagnostics(posterior)   # all metrics at once
add_rhat(posterior)               # R-hat only
add_ess(posterior)                # ESS only
add_mcse(posterior)               # MCSE only
```

Results are written under:

```text
posterior._auxiliary/diagnostics/mcmc/
```

Users should normally read them back through `posterior.diagnostics`, which
returns structured views such as `MCMCView`.


In [7]:
add_mcmc_diagnostics(posterior)

mcmc = posterior.diagnostics.mcmc   # MCMCView

print("R-hat:     ", mcmc.rhat)
print("ESS bulk:  ", mcmc.ess_bulk)
print("ESS tail:  ", mcmc.ess_tail)
print("MCSE mean: ", mcmc.mcse_mean)
print()

if mcmc.warnings:
    for w in mcmc.warnings:
        print("⚠", w)
else:
    print("✓ All MCMC diagnostics passed.")

R-hat:      {'intercept': NotComputed('R-hat requires at least 2 chains'), 'slope': NotComputed('R-hat requires at least 2 chains')}
ESS bulk:   {'intercept': 433.10269057959124, 'slope': 410.05555168331085}
ESS tail:   {'intercept': 480.608713116952, 'slope': 516.254330481698}
MCSE mean:  {'intercept': 0.0028643168356568727, 'slope': 0.0023220882008709773}

✓ All MCMC diagnostics passed.


In [8]:
# Or use the built-in formatted table
print()
print(posterior.diagnostics.summary_table())


Parameter     R-hat     ESS bulk    ESS tail    MCSE mean   
------------------------------------------------------------
intercept     N/A       433.1027    480.6087    0.0029      
slope         N/A       410.0556    516.2543    0.0023      

✓  All available diagnostics passed.



## 5. LOO-PSIS

`add_loo` computes PSIS-LOO and writes results under:

```text
posterior._auxiliary/diagnostics/runs/loo/
```

Read results through `posterior.diagnostics.loo`, a `LOOView`.

`add_loo(posterior)` first checks for pointwise
log likelihoods under:

```text
posterior._auxiliary/arviz/log_likelihood
```

If that group already exists, `add_loo` can call ArviZ directly. If it is
missing, pass `model=` and `data=` so `add_loo` computes pointwise log
likelihoods internally before running ArviZ.

The key things to show here are `elpd_loo`, `looic`, Pareto-k summaries, and the
warnings produced when Pareto-k values indicate unreliable LOO estimates.


In [10]:
add_loo(posterior, model=model_poisson, data=data)

loo = posterior.diagnostics.loo
print("ELPD-LOO:     ", loo.elpd_loo)
print("SE:           ", loo.se)
print("LOO-IC:       ", loo.looic)
print("Pareto-k max: ", loo.pareto_k_max)
print()
if loo.warnings:
    for w in loo.warnings:
        print("WARNING:", w)
else:
    print("LOO diagnostics passed.")


ELPD-LOO:      NotComputed('value is NaN')
SE:            32.06032022155848
LOO-IC:        NotComputed('value is NaN')
Pareto-k max:  1.0013096905292953



/Users/ylim2/miniconda3/envs/probpipe/lib/python3.13/site-packages/arviz_stats/loo/helper_loo.py:1146: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.67 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
